# Part 2 – Air Quality: 02 Clean & Feature Engineering

## Comprehensive Module Overview

Welcome to the **Transformation Layer** of our data pipeline. Having extracted our data as raw JSON in `01_ingest.ipynb`, this notebook focuses entirely on **Cleaning, Quality Assurance (QA), and Feature Engineering**.

Raw sensor data is notoriously messy. Sensors go offline, APIs return duplicates, and different systems use different timezone standards. A robust data pipeline does not just assume data is correct; it codifies a strict set of rules to filter out noise, filling or dropping anomalies logically, and joining datasets carefully to avoid Cartesian explosions or mismatched dates.

### End-to-End Workflow
1. **Load Raw Data**: Convert nested, hierarchical JSON structures from OpenAQ into a tabular Pandas DataFrame.
2. **Apply QA Rules**: Enforce our strict 4-Rule Quality Assurance protocol to drop unreliable or duplicate PM2.5 readings.
3. **Aggregate Weather Data**: Read Open-Meteo hourly weather, handle complex daylight saving time (DST) shifts, convert to UTC, and aggregate to daily metrics (mean temperature, mean wind speed, total precipitation).
4. **Relational Merge**: Join the Air Quality and Weather dataframes exactly on `(city, UTC Date)`.
5. **Feature Engineering**: Extract human-readable calendar features (`weekday`, `is_weekend`) for later categorical analysis.
6. **Final Assertions & Storage**: Algorithmically assert that no nulls or duplicates survived, then write the pristine dataset out as an **Apache Parquet** file.


## Environment Setup & Best Practices

We are heavily utilizing `pandas` for vectorized, in-memory data transformations. By suppressing Pandas' `FutureWarning` alerts (which often trigger on upstream library implementations of timezone mechanics), we keep our notebook output clean and readable without ignoring critical runtime errors.


In [ ]:
import json
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

## Configuration: Paths & Timezones

We declare our paths relatively to the repository root. Note the `CITIES` dictionary: we explicitely attach the IANA Timezone string (e.g. `America/New_York`) to each city. Because the Open-Meteo API returned data locally to each city, we will need this exact string to localize the naive datetimes before we can safely convert them to a universal UTC standard.


In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
ROOT    = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'part2_airquality' else Path(os.getcwd())
RAW_AQ  = ROOT / 'data' / 'raw' / 'openaq'
RAW_WX  = ROOT / 'data' / 'raw' / 'openmeteo'
CURATED = ROOT / 'data' / 'curated'
CURATED.mkdir(parents=True, exist_ok=True)

CITIES = {
    'nyc':     {'label': 'New York City', 'tz': 'America/New_York'},
    'la':      {'label': 'Los Angeles',   'tz': 'America/Los_Angeles'},
    'chicago': {'label': 'Chicago',       'tz': 'America/Chicago'},
}

## Step 1 – Load Raw OpenAQ JSON into a Flat Measurement Table

### The `load_flat` function
The OpenAQ daily summary JSON is deeply nested. Our custom `load_flat` function iterates through the `results` array of each city's JSON and extracts exactly the 5 fields we care about.

Crucially, we read `period.datetimeFrom.utc` as our primary timestamp. Working natively in UTC solves the 'DST double-hour' and 'DST missing-hour' problems that plague local-time analytics. We compile an initial DataFrame of approximately 270 rows, representing all 3 cities over the 90 days.


In [ ]:
def load_flat(path: Path, city_key: str) -> pd.DataFrame:
    """Expand a single OpenAQ JSON file into one row per daily measurement record."""
    with open(path) as f:
        raw = json.load(f)

    rows = []
    for r in raw['results']:
        rows.append({
            'city_key':        city_key,
            'city':            CITIES[city_key]['label'],
            # UTC timestamp — used for deduplication and timezone conversion
            'ts_utc':          r['period']['datetimeFrom']['utc'],
            # Local timestamp string (carries offset, e.g. -05:00)
            'ts_local':        r['period']['datetimeFrom']['local'],
            'value':           r['summary']['avg'],          # daily mean PM2.5
            'q75':             r['summary']['q75'],
            'q98':             r['summary']['q98'],
            'observed_count':  r['coverage']['observedCount'],  # hourly readings in day
        })
    return pd.DataFrame(rows)


raw_frames = [load_flat(RAW_AQ / f'pm25_{k}_2025Q1.json', k) for k in CITIES]
raw_aq = pd.concat(raw_frames, ignore_index=True)

print(f'Total records loaded: {len(raw_aq)}')
raw_aq.head(3)

## QA Rule 1 – Remove Exact Duplicates

**Defense in Depth**: Even when querying pre-aggregated APIs, network retries or paginated overlapping can result in duplicated rows. We run `drop_duplicates(subset=['city', 'ts_utc', 'value'])`.

We use the UTC timestamp for deduplication. Two records with identical PM2.5 values for the same city at the same absolute UTC moment are redundant. Pandas `.drop_duplicates()` scans the dataset and leaves only the first occurrence.


In [ ]:
before = len(raw_aq)
raw_aq = raw_aq.drop_duplicates(subset=['city_key', 'ts_utc', 'value'])
after  = len(raw_aq)
removed_dupes = before - after

print(f'QA Rule 1 – Duplicate removal')
print(f'  Records before : {before}')
print(f'  Duplicates removed : {removed_dupes}')
print(f'  Records after  : {after}')

## QA Rule 2 – Normalise and Extract Canonical UTC Date

Working across three different timezones (Eastern, Central, Pacific) can cause analytical nightmares if you compare 'Monday' across cities without a shared baseline. 

By coercing our UTC timestamp strings to timezone-aware pandas datetime objects (`pd.to_datetime(utc=True)`), and then extracting `.dt.date`, we create a canonical, universal `date` column. This means when we join OpenAQ data to Open-Meteo data, we are matching on the exact same 24-hour planetary window.


In [ ]:
raw_aq['ts_utc_parsed'] = pd.to_datetime(raw_aq['ts_utc'], utc=True)
raw_aq['date_utc']      = raw_aq['ts_utc_parsed'].dt.normalize()   # midnight UTC

print('QA Rule 2 – Timezone conversion')
print('  All timestamps converted to UTC before date extraction.')
print(f'  Timezone used for daily aggregation: UTC')
print(f'  Sample UTC dates:')
print(raw_aq[['city', 'ts_local', 'ts_utc', 'date_utc']].head(4).to_string(index=False))

## QA Rule 3 – Minimum Coverage Threshold (< 12 Measurements → NaN)

### The Danger of Sparse Data
OpenAQ provides a `coverage.observedCount` column indicating how many hourly sensor readings made up the daily average. A complete day has 24 readings. If a sensor went offline and only recorded 3 hours (say, during rush hour), the 'daily average' would be wildly skewed unrepresentative of the entire day.

**The Rule**: We mandate a minimum of 12 hourly readings ($\ge$ 50% temporal coverage). If a row has fewer than 12 readings, we programmatically nuke its PM2.5 values by overriding them with `np.nan` (Not a Number).


In [ ]:
MIN_OBS = 12
low_coverage_mask = raw_aq['observed_count'] < MIN_OBS

print(f'QA Rule 3 – Low-coverage days (observedCount < {MIN_OBS})')
print(f'  Total low-coverage records: {low_coverage_mask.sum()}')
print()
per_city = raw_aq.groupby('city')['observed_count'].apply(lambda s: (s < MIN_OBS).sum())
for city, n in per_city.items():
    print(f'  {city:20s}: {n} low-coverage day(s)')

# Set pm25 fields to NaN for low-coverage days
raw_aq.loc[low_coverage_mask, 'value'] = np.nan
raw_aq.loc[low_coverage_mask, 'q75']   = np.nan
raw_aq.loc[low_coverage_mask, 'q98']   = np.nan

## Derive Interpolated Analytics: P90 Calculation

While OpenAQ gives us the daily mean, environmental exposure studies often care about the top x-percentile of pollution. OpenAQ gives us $q_{75}$ and $q_{98}$, but $P_{90}$ (the 90th percentile) is a standard epidemiological benchmark.

### Mathematical Interpolation
We write a helper function `safe_p90(q75, q98)` that uses **linear interpolation** to estimate the 90th percentile between the 75th and 98th:
$$ P_{90} = q_{75} + \frac{(90 - 75)}{(98 - 75)} \times (q_{98} - q_{75}) $$
$$ P_{90} = q_{75} + \frac{15}{23} \times (q_{98} - q_{75}) $$

The function uses a `try/except` block and checks for `pd.isna()` to ensure that if QA Rule 3 blanked out the data, we don't throw an error, we just safely return `NaN`.


In [ ]:
def safe_p90(row):
    if pd.isna(row['q75']) or pd.isna(row['q98']):
        return np.nan
    return row['q75'] + (90 - 75) / (98 - 75) * (row['q98'] - row['q75'])

raw_aq['pm25_mean'] = raw_aq['value'].round(4)
raw_aq['pm25_p90']  = raw_aq.apply(safe_p90, axis=1).round(4)

aq_daily = (
    raw_aq[['city', 'date_utc', 'pm25_mean', 'pm25_p90', 'observed_count']]
    .rename(columns={'date_utc': 'date'})
    .sort_values(['city', 'date'])
    .reset_index(drop=True)
)

print(f'AQ daily table shape: {aq_daily.shape}')
aq_daily.head(4)

## QA Rule 4 – Drop Days with Missing PM2.5 (No Interpolation)

We drop any row containing `NaN` in the `pm25_mean` column (which catches the days we disqualified in Rule 3). 

**Why drop instead of interpolating the missing days?**
Missing air quality data isn't like a missing pixel in an image; weather and pollution are highly chaotic. Imputing a missing Tuesday using Monday and Wednesday's data is scientifically risky if a major weather front moved through. Given we only expect a tiny fraction of missing sensor days over a 90-day window, dropping the day is the most statistically honest choice.


In [ ]:
missing_mask = aq_daily['pm25_mean'].isna()

print('QA Rule 4 – Days dropped due to missing PM2.5 (no interpolation)')
dropped_per_city = aq_daily[missing_mask].groupby('city').size()

for city in [c['label'] for c in CITIES.values()]:
    n = dropped_per_city.get(city, 0)
    print(f'  {city:20s}: {n} day(s) dropped')

print(f'\n  Total days dropped: {missing_mask.sum()}')

aq_daily = aq_daily[~missing_mask].reset_index(drop=True)
print(f'  AQ daily rows remaining: {len(aq_daily)}')

## Weather Parsing & Hourly-to-Daily Aggregation

This is the most complex block in the transformation layer.
1. **Un-pivoting JSON**: We tear open the Open-Meteo `hourly` arrays and map them into a Pandas DataFrame. We explicitly set the types to floats.
2. **Advanced Timezone Mechanics**: The open-meteo timestamps are 'naive' (meaning they have no timezone information, they are just numbers on a clock) but we know from the API that they represent local city time. We use `.dt.tz_localize(tz, ambiguous='infer', nonexistent='shift_forward')`. This powerfully handles Daylight Saving Time edges (like 'Spring Forward' where 2 AM doesn't exist, and 'Fall Back' where 1 AM happens twice).
3. **Alignment to UTC**: Now that Pandas knows the exact local time, we convert it to UTC using `.dt.tz_convert('UTC')`. We strip it to just `.dt.date`.
4. **GroupBy Aggregation**: We have 24 rows per day. We group by `city` and `date`, taking the `.mean()` of temperature and wind speed, and the `.sum()` of hourly precipitation to get total daily rainfall.


In [ ]:
def parse_openmeteo(path: Path, city_label: str, tz: str) -> pd.DataFrame:
    with open(path) as f:
        raw = json.load(f)

    hourly = raw['hourly']
    df = pd.DataFrame({
        'datetime':        pd.to_datetime(hourly['time']).tz_localize(tz, ambiguous='infer', nonexistent='shift_forward'),
        'temperature_2m':  hourly['temperature_2m'],
        'windspeed_10m':   hourly['windspeed_10m'],
        'precipitation':   hourly['precipitation'],
    })
    # Convert to UTC and extract date — matches Rule 2
    df['date'] = df['datetime'].dt.tz_convert('UTC').dt.normalize()

    daily = (
        df.groupby('date')
          .agg(
              temp_mean       = ('temperature_2m', 'mean'),
              wind_speed_mean = ('windspeed_10m',  'mean'),
              precip_sum      = ('precipitation',  'sum'),
          )
          .reset_index()
    )
    daily.insert(0, 'city', city_label)
    daily[['temp_mean', 'wind_speed_mean', 'precip_sum']] = \
        daily[['temp_mean', 'wind_speed_mean', 'precip_sum']].round(4)
    return daily


wx_frames = []
for key, cfg in CITIES.items():
    df = parse_openmeteo(RAW_WX / f'weather_{key}_2025Q1.json', cfg['label'], cfg['tz'])
    wx_frames.append(df)
    print(f"{cfg['label']:20s}: {len(df)} daily weather rows")

wx_all = pd.concat(wx_frames, ignore_index=True)
print(f'Total weather rows: {len(wx_all)}')

## Relational Merge: PM2.5 ∩ Weather

We use `pd.merge()` to horizontally smash our PM2.5 dataframe and our Weather dataframe together. 

We use an `how='inner'` join on the composite key `['city', 'date']`. This operates exactly like an SQL `INNER JOIN`. If a date exists in the Weather data but was dropped from the PM2.5 data (due to QA Rule 4), the inner join will naturally discard that Weather row. This guarantees our final dataset only contains days with **both** valid PM2.5 and Weather readings.


In [ ]:
merged = aq_daily.merge(wx_all, on=['city', 'date'], how='inner')
print(f'Merged rows: {len(merged)}')
merged.head(3)

## Feature Engineering: Categorical Calendar Variables

Machine learning models and statistical analysis often segment data by human calendar constructs, looking for anthropogenic (human-caused) cycles.
- **`weekday`**: Extracted directly using Pandas `.dt.day_name()`. Yields `Monday`, `Tuesday`, etc.
- **`is_weekend`**: A binary flag (1 for weekend, 0 for weekday). We use the `.dt.dayofweek` property where Monday=0 and Sunday=6. We use `.isin([5,6])` mapped to an integer.


In [ ]:
merged['weekday']    = merged['date'].dt.day_name()
merged['is_weekend'] = merged['date'].dt.dayofweek.isin([5, 6]).astype(int)

COLS = ['city', 'date', 'pm25_mean', 'pm25_p90',
        'temp_mean', 'wind_speed_mean', 'precip_sum',
        'weekday', 'is_weekend']

curated = merged[COLS].sort_values(['city', 'date']).reset_index(drop=True)
print(f'Final curated shape: {curated.shape}')
curated.dtypes

## Final Quality Assertions: Trust, but Verify

Before we release this dataset to the downstream analytical layer, we programmatically assert its mathematical purity. The python `assert` keyword will throw a loud, terminal `AssertionError` if the condition is false.
1. **Primary Key Uniqueness**: `len(df) == len(df.drop_duplicates(subset=['city', 'date']))`. There should be exactly one row per city per day.
2. **No Nulls Allowed**: `df.isna().sum().sum() == 0`. We shouldn't be passing NaNs downstream.
3. **Physical Reality Check**: `(df['pm25_mean'] >= 0).all()`. You cannot have negative particulate concentration. If the sensor glitched and reported -4 µg/m³, this catches it.


In [ ]:
# 1. No duplicate (city, date) pairs
dupes = curated.duplicated(subset=['city', 'date']).sum()
assert dupes == 0, f'{dupes} duplicate (city, date) pairs remain!'

# 2. No NaN values in any column
nulls = curated.isnull().sum()
assert nulls.sum() == 0, f'Null values remain:\n{nulls[nulls>0]}'

# 3. pm25_mean > 0 everywhere
assert (curated['pm25_mean'] > 0).all(), 'Non-positive pm25_mean detected!'

print('All quality assertions passed.')
curated.describe().round(2)

## Sink to Storage: Curated Parquet

The dataset has passed all inspections. We sink it to disk as an **Apache Parquet** file. Unlike CSVs which represent everything as heavy text strings, Parquet is a compressed columnar binary format. It intrinsically remembers that the `date` column is a `datetime` object and `is_weekend` is an `integer`. This saves the next notebook from having to guess or re-parse datatypes.


In [ ]:
out_path = CURATED / 'part2_airquality_curated.parquet'
curated.to_parquet(out_path, index=False)
print(f'Saved  : {out_path}')
print(f'Size   : {out_path.stat().st_size / 1024:.1f} KB')
print(f'Rows   : {len(curated)}')
print(f'Columns: {list(curated.columns)}')

## Executive QA Audit Report

We explicitly print out the lifecycle of the data volume. Data engineers should always inspect how many rows entered the pipeline versus how many survived the QA gauntlet to ensure the rules aren't being overly aggressive.


In [ ]:
print('=' * 55)
print('QA SUMMARY')
print('=' * 55)
print(f'Rule 1 – Exact duplicates removed      : {removed_dupes}')
print(f'Rule 2 – Timezone for aggregation      : UTC')
print(f'Rule 3 – Low-coverage days (< {MIN_OBS} obs)  : {low_coverage_mask.sum()}')
for city in [c['label'] for c in CITIES.values()]:
    n = dropped_per_city.get(city, 0)
    print(f'Rule 4 – Days dropped ({city:13s}): {n}')
print(f'Rule 4 – Total days dropped            : {missing_mask.sum()}')
print('=' * 55)
print(f'Final curated rows                     : {len(curated)}')